# ML-04 — Search Intelligence Data Contract (my lane: Refresh / Content Opportunity Scoring)

This notebook writes down **what one row means** for my lane, **which tables** it uses, **which
time windows**, **what it predicts**, and **one thing it deliberately excludes** — then proves the
claims with real queries on the warehouse, builds a five-feature frame, and runs the leakage trap
on purpose so I can see it, then remove it.

> Skill router: loaded `writing-data-contracts` + `flyrank/flyrank-data` (per `skills/README.md`).
> Lane continuity: `w02_ml_task_framing.ipynb` framed Lane 2 (Refresh / Content Opportunity
> Scoring) as a **ranking task** with a future-window proxy label; this notebook is that lane's
> data contract on the real warehouse.

**Iteration rule I follow:** I develop on a **mid-panel month** (`month = 2026-03` for features,
`2026-04` for the outcome). The `_sample` table is exactly the final month (June 2026) — I treat it
as a sealed test month and never develop label logic on it.

## 0. Setup — connect to the warehouse

The token comes from the `HF_TOKEN` environment variable (Colab Secret) or a local `.env` file.
It is **never printed and never pasted in a cell**.

In [1]:
%pip -q install duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    # Local fallback: walk up from the notebook's folder to find .env at the repo root.
    import pathlib
    for cand in pathlib.Path.cwd().parents:
        p = cand / ".env"
        if p.exists():
            for line in p.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("HF_TOKEN="):
                    HF_TOKEN = line.strip().split("=", 1)[1].strip().strip('"').strip("'")
            if HF_TOKEN:
                break
assert HF_TOKEN, "No HF_TOKEN found - set it as an env var / Colab Secret, or a .env file with HF_TOKEN=hf_..."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
F3 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"  # feature month
F4 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"  # outcome month
DC = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected. Feature month = 2026-03 | outcome month = 2026-04 | dims = dim_content")

Connected. Feature month = 2026-03 | outcome month = 2026-04 | dims = dim_content


## 1. The contract, in plain words (5 answers)

**1. What one row means — one content page.** The daily fact table holds one row per page per day
(`report_date x client x content`). My lane collapses that to **one row = one pseudonymized
content page** (`content_hash_id`) with its measurements for the decision month. Every score,
feature, and label is a property of *a page*.

**2. Which table(s).** `fact_content_daily_performance` (month `2026-03` = feature month,
month `2026-04` = outcome month) joined to `dim_content` for the page's static age metadata.

**3. Which time window.** Features come from **2026-03-01 → 2026-03-31** (the feature month); the
label is measured in **2026-04-01 → 2026-04-30** (the outcome month). The decision moment is
**the end of 2026-03-31** — an editor reviews the queue on 1 April, using only what March showed.
The feature window and the label window never overlap.

**4. What I predict / rank — a future proxy label.** `will_decline`: the page's April impressions
are more than 20% **below** its March impressions (measured on pages with March base demand
≥ 100 impressions). It is a *proxy* — it flags pages that measurably lost visibility after the
decision point; it is not a promise that a refresh would have recovered them. I rank pages by
this risk so an editor spends the first reviews on the likeliest decliners.

**5. One thing I deliberately exclude — `fact_content_query_90d`.** Its fixed 90-day window
(the final ~3 months of the snapshot) **overlaps the label period** for a mid-panel decision.
Using its `impressions_90d` / `*_last30` columns would smuggle the answer into the features.

## 2. Fields: feature / label / context / excluded

Every field I touch goes in exactly one bucket. The model may learn from **features** only.

In [3]:
classification = pd.DataFrame([
    ("log_imp_mar",   "feature", "log1p(March GSC impressions) - volume/momentum of the page"),
    ("ctr_mar",       "feature", "March clicks / March impressions - click-worthiness of the listing"),
    ("pos_mar",       "feature", "March average GSC position (lower = higher on the page)"),
    ("days_mar",      "feature", "days in March the page drew >= 1 impression - consistency of presence"),
    ("age_days",      "feature", "page age in days at the decision moment (from dim_content)"),
    ("will_decline",  "label",   "April impressions < 80% of March impressions - the thing I predict"),
    ("content_hash_id", "context", "pseudonym - grouping/joining only, never learned from"),
    ("client_hash_id",  "context", "pseudonym - grouping/client-splits only, never learned from"),
    ("report_date",   "context", "calendar date - windowing only"),
    ("fact_content_query_90d table", "excluded",
     "fixed 90-day window overlaps the label period -> leakage"),
    ("GA4 columns before ga4_data_available", "excluded",
     "zero-FILLED before GA4 tracking starts - zeros there are not 'no engagement'"),
    ("June 2026 (the _sample / final month)", "excluded",
     "sealed test month - label logic developed there would be tuned on the answer"),
], columns=["field", "bucket", "why"])

print(classification.to_string(index=False))

                                field   bucket                                                                          why
                          log_imp_mar  feature                   log1p(March GSC impressions) - volume/momentum of the page
                              ctr_mar  feature           March clicks / March impressions - click-worthiness of the listing
                              pos_mar  feature                      March average GSC position (lower = higher on the page)
                             days_mar  feature        days in March the page drew >= 1 impression - consistency of presence
                             age_days  feature                   page age in days at the decision moment (from dim_content)
                         will_decline    label           April impressions < 80% of March impressions - the thing I predict
                      content_hash_id  context                        pseudonym - grouping/joining only, never learned from
        

## 3. Verify it with queries — three facts, on the mid-panel month 2026-03

A contract line without a query next to it is a guess. Three facts, three queries:

1. **Grain** — one row really is what I said (daily grain holds; aggregation collapses to one row per page).
2. **Row count + date span** — how big my slice is and over which dates.
3. **Availability** — filter with `IS TRUE` and count how many rows survive.

### Fact 1 — the grain

In [4]:
# Grain probe: a (report_date, client, content) triple must be unique in the daily fact.
dups = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM (SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
          FROM {F3} GROUP BY 1, 2, 3 HAVING c > 1)
""").fetchone()[0]
print(f"daily-grain violations (report_date x client x content with COUNT>1): {dups}  -> grain holds")

# My lane grain: after aggregation, one row per content page.
agg = con.sql(f"""
    SELECT COUNT(*) AS rows_after, COUNT(DISTINCT content_hash_id) AS distinct_pages
    FROM (SELECT content_hash_id FROM {F3} GROUP BY content_hash_id)
""").fetchone()
print(f"after aggregation to my lane grain: {agg[0]:,} rows, {agg[1]:,} distinct pages -> one row per page holds")

daily-grain violations (report_date x client x content with COUNT>1): 0  -> grain holds


after aggregation to my lane grain: 331,437 rows, 331,437 distinct pages -> one row per page holds


### Fact 2 — my slice's row count and date span

In [5]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT report_date)     AS n_days,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM {F3}
""").fetchone()
print(f"month=2026-03 slice: {span[0]:,} daily rows")
print(f"  across {span[1]} days, {span[2]} clients, {span[3]:,} content pages")
print(f"  date span: {span[4]} -> {span[5]}")

month=2026-03 slice: 9,841,378 daily rows
  across 31 days, 55 clients, 331,437 content pages
  date span: 2026-03-01 -> 2026-03-31


### Fact 3 — availability (filter with `IS TRUE`)

Rows before a client's GA4 start are zero-filled with `ga4_data_available = FALSE`, and rows
without search tracking carry `gsc_data_available = FALSE`. Filtering on the flag separates
"measured zero" from "not tracked at all".

In [6]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_ok,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_ok
    FROM {F3}
""").fetchone()
print(f"total rows in month=2026-03             : {avail[0]:>12,}")
print(f"rows with gsc_data_available IS TRUE    : {avail[1]:>12,}  ({avail[1]/avail[0]:.1%} survive)")
print(f"rows with ga4_data_available IS TRUE    : {avail[2]:>12,}  ({avail[2]/avail[0]:.1%} survive)")

total rows in month=2026-03             :    9,841,378
rows with gsc_data_available IS TRUE    :    3,611,061  (36.7% survive)
rows with ga4_data_available IS TRUE    :      413,966  (4.2% survive)


### 3b. The five-feature frame (max five) — knowable at the decision moment

Decision moment = **2026-04-01** (right after March closed, before any April data accrues).
Each feature is knowable at that moment because…

In [7]:
# Build the lane's feature frame from month=2026-03, one row per page.
features = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS imp_mar,
           SUM(gsc_clicks)      AS clk_mar,
           AVG(gsc_avg_position) AS pos_mar,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_mar,
           COUNT(*)             AS rows_mar,
           SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_rows_mar
    FROM {F3}
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 100
       AND SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) > 0
""").df()

meta = con.sql(f"SELECT content_hash_id, content_created_date FROM {DC}").df()
features = features.merge(meta, on="content_hash_id", how="left")
features["age_days"]   = (pd.Timestamp("2026-03-31") - pd.to_datetime(features["content_created_date"])).dt.days
features["ctr_mar"]    = features["clk_mar"] / features["imp_mar"]
features["log_imp_mar"] = np.log1p(features["imp_mar"])

feat_cols = ["log_imp_mar", "ctr_mar", "pos_mar", "days_mar", "age_days"]
print(f"feature frame: {len(features):,} pages x {features.shape[1]} columns (one row per page)")
features[feat_cols].describe().round(3)

feature frame: 101,441 pages x 11 columns (one row per page)


,log_imp_mar,ctr_mar,pos_mar,days_mar,age_days
count,101441.000,101441.000,101441.000,101441.000,101441.000
mean,6.819,0.003,14.439,28.500,188.937
std,1.412,0.004,14.285,5.022,127.259
min,4.615,0.000,0.014,1.000,1.000
25%,5.649,0.000,4.990,29.000,71.000
50%,6.668,0.001,8.699,31.000,187.000
75%,7.830,0.004,19.178,31.000,265.000
max,13.333,0.156,93.693,31.000,494.000


| Feature | Knowable at the decision moment because… |
|---|---|
| `log_imp_mar` | March impressions — the feature month has fully closed by 2026-04-01. |
| `ctr_mar` | March clicks / March impressions — both March totals are final by the decision moment. |
| `pos_mar` | March average GSC position — every March position measurement is already in. |
| `days_mar` | days in March with ≥ 1 impression — March is fully observed by the decision moment. |
| `age_days` | page age at the decision moment — static content metadata known long before. |

### 3c. The label — a future outcome, measured after the decision moment

`will_decline = April impressions < 80% of March impressions`, counted only on pages with March
base demand ≥ 100. No feature window overlaps it.

In [8]:
apr = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS imp_apr,
           SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END)               AS gsc_rows_apr
    FROM {F4}
    GROUP BY content_hash_id
""").df()

frame = features.merge(apr, on="content_hash_id", how="left")
frame["imp_apr"]       = frame["imp_apr"].fillna(0)
frame["gsc_rows_apr"]  = frame["gsc_rows_apr"].fillna(0)
frame["apr_mar_ratio"] = frame["imp_apr"] / frame["imp_mar"]
frame["will_decline"]  = (frame["imp_apr"] < 0.8 * frame["imp_mar"]).astype(int)

print(f"labeled frame: {len(frame):,} pages | label rate = {frame['will_decline'].mean():.1%} "
      f"(n_pos = {frame['will_decline'].sum():,})")
print(f"pages with no GSC rows in April (counted as 0): {(frame['gsc_rows_apr'] == 0).sum():,} "
      f"({(frame['gsc_rows_apr'] == 0).mean():.1%} of frame)")
frame[["imp_mar", "imp_apr", "apr_mar_ratio", "will_decline"]].head()

labeled frame: 101,441 pages | label rate = 51.7% (n_pos = 52,492)
pages with no GSC rows in April (counted as 0): 548 (0.5% of frame)


,imp_mar,imp_apr,apr_mar_ratio,will_decline
0,925.0,0.0,0.000000,1
1,978.0,295.0,0.301636,1
2,197.0,39.0,0.197970,1
3,8968.0,3853.0,0.429639,1
4,247.0,137.0,0.554656,1


### 3d. The trap — ONE label-derived column, on purpose

Quick score: logistic regression on the five features (fixed seed 42, 80/20 split), measured with
ROC AUC + Precision@50 (the lane's metric — the front of the queue must be right).

Then I add **one label-derived column on purpose**: `apr_mar_ratio` — the very ratio the label
thresholds (`will_decline = apr_mar_ratio < 0.8`). It is April's outcome, measured after the
decision moment. If my score jumps toward perfect, that column leaked the answer. Then I delete
it and keep the honest number — the leakage lesson from notebook 02, on real warehouse data.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

FEATS = ["log_imp_mar", "ctr_mar", "pos_mar", "days_mar", "age_days"]
model_frame = frame.copy()
X = model_frame[FEATS].copy()
y = model_frame["will_decline"].values

def quick_score(X, y, seed=42):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, p)
    p50 = y_te[np.argsort(p)[::-1][:50]].mean()
    return auc, p50

auc0, p500 = quick_score(X, y)
print("HONEST - 5 features, no leakage:")
print(f"   ROC AUC = {auc0:.4f} | Precision@50 = {p500:.1%} ({int(round(p500 * 50))}/50)")

leak = X.copy()
leak["apr_mar_ratio"] = model_frame["apr_mar_ratio"].values
auc1, p501 = quick_score(leak, y)
print("\nLEAKED - same model + ONE label-derived column (April/March impression ratio):")
print(f"   ROC AUC = {auc1:.4f} | Precision@50 = {p501:.1%} ({int(round(p501 * 50))}/50)")

auc2, p502 = quick_score(X, y)
print("\nCLEAN - leaked column dropped, honest score restored:")
print(f"   ROC AUC = {auc2:.4f} | Precision@50 = {p502:.1%} ({int(round(p502 * 50))}/50)")
print("\nLesson: a single column from the label window turned a weak model 'perfect'.")

HONEST - 5 features, no leakage:
   ROC AUC = 0.6213 | Precision@50 = 60.0% (30/50)



LEAKED - same model + ONE label-derived column (April/March impression ratio):
   ROC AUC = 1.0000 | Precision@50 = 100.0% (50/50)



CLEAN - leaked column dropped, honest score restored:
   ROC AUC = 0.6213 | Precision@50 = 60.0% (30/50)

Lesson: a single column from the label window turned a weak model 'perfect'.


## 4. Data limits — one named limitation (and friends)

**Named limitation — "absent" counts as zero.** Pages that have no GSC rows in April
(548 of 101,441, 0.5% of the frame) are treated as zero April impressions. Absence can be a real
drop-to-nothing, or it can be a tracking gap — this data alone cannot tell them apart, so a few
labels may flag a tracking gap instead of a decline.

Other limits to say out loud:

- **Directional, not causal.** `will_decline` is an *observed* drop in visibility. This data does
  not prove a refresh would have recovered the page — that needs an experiment or causal design.
- **GSC-only early history.** March rows with `gsc_data_available = FALSE` are excluded from
  features; GA4 columns before `ga4_data_available` are zero-filled and never used as features.
- **Unbalanced panel.** Clients start tracking on different dates; my slice is one calendar month,
  so pages from newer clients carry no earlier history for trend features.
- **Proxy, not the true outcome.** A page declining by volume is a decision-support signal for
  review — not proof of a Google ranking loss or of a good refresh candidate.

In [10]:
# The numbers behind the named limitation.
n_absent = int((frame['gsc_rows_apr'] == 0).sum())
print(f"pages with no GSC rows in April (counted as zero): {n_absent:,} of {len(frame):,} "
      f"({n_absent/len(frame):.1%})")

pages with no GSC rows in April (counted as zero): 548 of 101,441 (0.5%)


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Contract, one line:** one row = one content page; features from March 2026 + static page age;
label = April 2026 impressions < 80% of March (base demand ≥ 100); ranking risk for a refresh
review queue; `fact_content_query_90d` deliberately excluded for window overlap.